# 第68课 · 100 帧如何吐出 5 个字——CTC 对齐：blank 符号与标签折叠（label collapse）

**目标**：理解 CTC——blank + 折叠 + 多路径，解决「100 帧如何吐出 5 个字」。

> **痛点开场**：不知道边界时怎么对齐。**图优先**于公式；贪婪解码建立路径压缩直觉。

🔗 **Aurora 连接**：`aurora.speech` 将调用 `torch.nn.CTCLoss`；读懂 CTC 是理解 Whisper 训练目标的前提。 本节手写贪婪解码，建立对 CTC 路径压缩的直觉。

← **上一课**　[L67 · Edit Distance 从零实现](L67_edit_distance.ipynb)

> 上节课学习了 **Edit Distance 从零实现**：Levenshtein 动态规划，WER 的数学基础。  
> 本课将探讨 **CTC 对齐原理**。

## 本课剧情：模型每秒输出 100 个字符，但句子只有 5 个字——这合法吗？

语音识别模型每帧输出一个 token 预测——对于 1 秒音频有约 100 帧。但目标文字可能只有 5 个字符。模型怎么把"100 个预测"压缩成"5 个字符"？

**CTC（Connectionist Temporal Classification）的答案**：引入 blank 符号 `∅`。

想象你在用手机打字，但键盘有延迟，你按了 `a-a-a-∅-b-b-∅-c`，手机输出 `abc`——这就是 CTC。

**CTC 路径 → 目标序列，两步压缩**：
```
路径 π  = [a, a, ∅, b, b, ∅, ∅, c]   （模型帧级输出，含重复和blank）
步骤1   → [a, ∅, b, ∅, ∅, c]           去相邻重复（a-a→a, b-b→b）
步骤2   → [a, b, c]                     去blank
```

**CTC 为什么有效**：一个目标序列 `"abc"` 对应**指数级**的路径数（各种blank位置组合都合法），模型对所有这些路径的概率求和就是输出序列的总概率。训练时最大化这个总概率，不需要预先标注"哪帧对应哪个字符"。

**贪婪解码（greedy decode）**：每帧取概率最大的 token，然后做压缩。快速，但不保证最优：

```
logits (T, V) → argmax每帧 → 路径π → collapse() → 目标序列
```

本节任务：实现 `ctc_greedy_decode(logits, blank=0)`，验证在已知 logits 上正确输出。

## 🤔 为什么工程师要发明它？(Why did engineers invent this?)

- **不用它会怎样？** 你得先人工标注"第 37 帧对应字母 h、第 38 帧还是 h、第 39 帧是 e……"——一秒音频 100 帧全靠手对齐，成本高到无法规模化训练。
- **它解决了什么真实问题？** 语音**帧数远多于字符数**，且没有逐帧标注。CTC 让你只给"这段音频 = 这句话"（序列级标注），模型自己去猜每个字对齐到哪些帧。
- **后面哪里还会再用到？** L69（前向算法把这些路径求和成损失）、L70+（Whisper 虽改用交叉熵，但"帧多字少"的对齐难题是同一个）。

### 最小例子：many-to-one（多帧 → 少字）

模型每帧吐一个符号（`_` = blank）。5 帧输出 → collapse（**先去相邻重复，再去 blank**）→ 2 个字：

```
A A _ B B   →  去重复→ A _ B  →  去blank→  A B   ✅
```

**关键直觉**：很多不同的帧序列都会 collapse 成同一个 `"AB"`——这就是 **many-to-one**：

```
A A _ B B   →  AB
_ A _ B _   →  AB
A _ _ _ B   →  AB
A A A B B   →  AB   （AA→A，BB→B）
```

训练时不挑其中某一条，而是把**所有** collapse 成 `"AB"` 的路径概率**加起来**（这正是 L69 要做的事）。

## ⚠️ 常见误解 (Common Pitfall)

> 不要把 blank `_` 理解成"空字符 / 空格 / 静音"。它是 **"这一刻先不吐字"** 的占位符——一个纯粹的**对齐工具**。它的两个作用：(1) 让字符数少于帧数时有地方"填空"；(2) 分隔两个相同字符——`A _ A` collapse 成 `AA`（两个 A），而 `A A` 只 collapse 成一个 `A`。


## 序列级标注 vs 帧级标注：为什么不需要对齐？

**传统方法（需要帧级标注，成本高）**：

```
音频波形（1秒=100帧）：    [──── ──── ──── ...]
人工标注（逐帧）：        h    o    l    l    o    ∅    ∅    ...
每一帧都要标注属于哪个字母，工作量巨大（100帧都要标）
```

**CTC 的聪明之处（只需序列级标注，低成本）**：

```
音频波形（1秒=100帧）：    [──── ──── ──── ...]
序列级标注（整体）：                "hello"
模型自己猜对齐！
```

**为什么行得通**？——因为 CTC 用"路径的总概率和"来编码对齐的**所有可能性**：

```
目标 = "ab"

模型输出：3帧的概率分布
  帧1：a=80%, ∅=20%
  帧2：b=70%, ∅=30%
  帧3：∅=60%, b=40%

合法路径（压缩后都是"ab"）：
  [a, b, ∅]  概率 = 0.8 × 0.7 × 0.6 = 0.336
  [a, ∅, b]  概率 = 0.8 × 0.3 × 0.4 = 0.096
  [a, b, b]  概率 = 0.8 × 0.7 × 0.4 = 0.224
  ...
  
P(y="ab" | 音频) = Σ 所有合法路径的概率
                = 0.336 + 0.096 + 0.224 + ...
                ≈ 0.8（假设算出来）
```

**模型如何学习**：
- 训练时，最大化 P(y="ab" | x=音频)
- 这相当于让模型给所有能导出"ab"的对齐分配概率
- 模型自动学到：某个"ab"的真实对齐可能是 [a,b,∅]，也可能是 [a,∅,b]，都合理
- 不需要人工指定"a在帧1"、"b在帧2"这样的硬约束

**效果对比**：
| 方法 | 标注成本 | 对齐灵活性 | 训练数据需求 |
|---|---|---|---|
| 帧级标注 | 高（逐帧手工）| 固定（只有一种对齐）| 少（约束强） |
| CTC（序列级） | 低（只标序列）| 灵活（所有合法对齐） | 多（学习空间大） |

这就是为什么 CTC 在语音识别领域革命性——大大降低了标注成本。

In [ ]:
import numpy as np

## 1. CTC 路径与路径概率

设输入序列长度为 `T`，词表（vocabulary）大小为 `V`（含 blank=0）。
模型在每一帧 `t` 输出一个概率分布 `p(token | t)`，形状 `(T, V)`。

**CTC 路径** `π` 是长度为 `T` 的 token 序列（每帧一个符号，含 blank）。
路径概率为各帧概率之积：

```
P(π) = ∏_{t=1}^{T}  p(π_t | t)
```

**CTC 损失** 最大化所有「压缩后等于目标 `y`」的路径的概率之和：

```
P(y | x) = Σ_{π : collapse(π)=y}  P(π)
```

路径数量是指数级的，实际用**前向-后向算法**动态规划（dynamic programming，DP）精确计算。

In [ ]:
# 演示：3帧，词表={blank=0, a=1, b=2}
# 构造一个简单的对数概率矩阵（手动指定）
T, V = 3, 3
log_probs = np.array([
    [np.log(0.1), np.log(0.8), np.log(0.1)],  # 帧0：a 最大
    [np.log(0.7), np.log(0.2), np.log(0.1)],  # 帧1：blank 最大
    [np.log(0.1), np.log(0.1), np.log(0.8)],  # 帧2：b 最大
])
# 列举路径 (a, blank, b) 和 (a, a, b) 两条路径，两者压缩后都是 [a, b]
path1 = [1, 0, 2]  # a ∅ b  -> collapse -> a b
path2 = [1, 1, 2]  # a a b  -> collapse -> a b
p_path1 = np.exp(log_probs[0,1] + log_probs[1,0] + log_probs[2,2])
p_path2 = np.exp(log_probs[0,1] + log_probs[1,1] + log_probs[2,2])
print(f'P(a ∅ b) = {p_path1:.4f}')
print(f'P(a a b) = {p_path2:.4f}')
print(f'P(y=[a,b] | x) ≈ {p_path1 + p_path2:.4f}  (只枚举了两条路径)')


## 2. 压缩规则（collapse）

给定一条路径 `π`（含 blank），压缩为目标序列的规则分两步：

```
步骤1：去掉相邻重复  [a, a, ∅, b, b, ∅, c] → [a, ∅, b, ∅, c]
步骤2：去掉 blank    [a, ∅, b, ∅, c]       → [a, b, c]
```

注意 blank 的作用之一是**分隔同字符**：
路径 `[a, a, b]` 压缩为 `[a, b]`；
路径 `[a, ∅, a, b]` 压缩为 `[a, a, b]`（两个 a 之间有 blank 则不合并）。

In [ ]:
# 传统 for 循环版本的 collapse（看不懂列表推导式的话，这就是你要的）
def collapse_with_loop(path, blank=0):
    """CTC 路径压缩：先去相邻重复，再去 blank（用 for 循环，逐步易懂）。
    
    Args:
        path: list[int]，帧级符号序列（含 blank）
        blank: blank 符号的 token id，默认 0
    Returns:
        list[int]，压缩后的序列（不含 blank，相邻重复已合并）
    """
    # ===== 第一步：去掉相邻重复 =====
    deduped = []
    for i, token in enumerate(path):
        # 只有当 i=0（第一个）或 当前 token ≠ 前一个 token 时，才保留
        if i == 0 or token != path[i - 1]:
            deduped.append(token)
    
    # ===== 第二步：去掉 blank =====
    final = []
    for token in deduped:
        if token != blank:
            final.append(token)
    
    return final

# 验证这个版本和之后的列表推导式版本效果一样
test_path = [1, 1, 0, 2, 2, 0, 3]
print(f"For 循环版本: collapse_with_loop({test_path}) = {collapse_with_loop(test_path)}")


## 为什么必须**先去相邻重复，再去 blank**？顺序能颠倒吗？

这是一个关键的设计选择。让我们用反例来看为什么顺序不能反：

```
路径 π = [a, ∅, a, b]   # 两个 a 中间隔着 blank

【正确顺序】先去重复 → 再去 blank：
  [a, ∅, a, b]  →  去相邻重复  →  [a, ∅, a, b]  (∅在中间，a和a没有相邻，保留)
              →  去 blank  →  [a, a, b]  ✓ 两个 a 存活

【错误顺序】先去 blank → 再去重复：
  [a, ∅, a, b]  →  去 blank  →  [a, a, b]
              →  去相邻重复  →  [a, b]  ✗ 两个 a 错误合并成一个！
```

**核心规则**：`∅` 的一个作用就是**分隔两个相同字符**，让它们不会被误合并。
- `[a, a, b]` → `[a, b]`（aa相邻，合并为一个a）
- `[a, ∅, a, b]` → 先去重复（∅保留，aa不相邻）→ `[a, ∅, a, b]`；再去blank → `[a, a, b]`（两个a）

所以 **collapse 的定义必须是"先去重复，再去blank"**。这个顺序不能改，改了会导致不同的压缩结果，训练目标就完全不同了。

### 路径概率：用乘法把各帧预测连接起来

模型在每一帧 `t` 上输出一个数字给每个 token，称为**逻辑值（logits）** 或 **得分**。
softmax 后就是概率。比如帧 3 的模型输出是 `[0.6, 0.3, 0.1]`，意思是"帧3预测 token-0 概率60%，token-1 概率30%，token-2 概率10%"。

**条件概率记号** `p(π_t | t)` 的意思很直白：
- 竖杠 `|` 就是"在……条件下"
- `p(π_t | t)` = "已知在第 `t` 帧，预测的符号是 π_t 的概率"
- 高中类比：`p(下雨 | 天阴)` = "天阴的时候下雨的概率"

一条路径 `π = [token₁, token₂, ..., tokenₜ]` 从**多个帧**拼接而成，各帧的预测是**独立发生**的（模型每帧单独看当前帧输入），所以总概率是各帧概率的**乘积**：

```
P(π) = p(π₁|帧1) × p(π₂|帧2) × ... × p(πₜ|帧T)
     = ∏_{t=1}^{T}  p(π_t | t)
```

记号说明：
- `∏` 就是连乘 `×`（求积）
- `Σ` 就是求和 `+`
- `∏_{t=1}^{T}` 意思是从帧 t=1 连乘到 t=T

### 为什么是**求和**而不是**取最大值**？

直觉上一个目标序列 `y`（比如"ab"）对应很多条不同的帧级路径。
最贪心的想法是"只选概率最高的那条路径"，即：
```
P_greedy(y|x) = max_{π: collapse(π)=y}  P(π)
```

但 CTC 的设计是：**把所有压缩成 y 的路径概率加起来**：
```
P_CTC(y|x) = Σ_{π: collapse(π)=y}  P(π)
```

**为什么求和而不是取 max？**——这是**概率的正确使用**：
- 想象掷骰子。"点数≥4"的概率 = P(4) + P(5) + P(6) = 1/6 + 1/6 + 1/6 = 1/2
- 不是 max(P(4), P(5), P(6)) = 1/6
- 如果一个目标序列能由多条路径导出（掷出4、5、6都算"≥4"），就应该把这些概率加起来

**图示对比**：
```
路径概率视角：
  路径1 [a,∅,b]  概率 0.3
  路径2 [a,a,b]  概率 0.2    } 都压缩成"ab"
  路径3 [a,∅,∅,b] 概率 0.1  }

贪心：P(y="ab") = max(0.3, 0.2, 0.1) = 0.3
CTC:  P(y="ab") = 0.3 + 0.2 + 0.1 = 0.6  ✓ 这才是"y="ab"发生"的真实概率
```

CTC 训练时最大化 P_CTC，就是说：**鼓励模型给所有合法路径分配高概率**，而不是赌一条路径。这样更稳健。

### 指数级路径数：为什么一个目标序列对应这么多条路径？

**具体例子**：目标 = `"ab"`，帧数 T=4，词表 V={0:∅, 1:a, 2:b}

合法的帧级路径有哪些？任何只要"压缩后是 ab"的都行：
```
[a, ∅, b, ∅]  → a b    ✓
[a, a, b, ∅]  → a b    ✓  (aa → a 后变成 ab)
[a, ∅, b, b]  → a b    ✓  (bb → b)
[a, ∅, ∅, b]  → a b    ✓
[a, a, ∅, b]  → a b    ✓
```

为什么这么多？因为：
1. **Blank 可以任意插入**：a 后面可以跟任意多个 blank 或其他 a（再去重）
2. **重复字符可以放在一起**：`aa` → `a`（去重后还是 a）

**路径数公式**：设输入 T 帧，词表 V 个符号（含 blank），任意一条路径都是 T 个 token 的序列。
每一帧独立选择，总共 **V^T** 条可能的帧级路径。其中大部分压缩后会得到不同的目标序列，但一个目标序列下可能有数百甚至数千条路径。

**数值例子**：
- T=5 帧，V=4（blank + 3个token）→ 4^5 = 1024 条总路径
- 其中可能有 50 条路径压缩成 "ab"，100 条压缩成 "abc"，等等
- 如果用贪心（只看一条最优路径）就错过了 49 条支持 "ab" 的次优路径

这就是为什么 **CTC 不能简单取 max**，而要对所有合法路径求和——每一条都有其贡献。

In [ ]:
# 实际计数：对于一个目标序列，有多少条合法路径？
from itertools import product

def count_ctc_paths(target_labels, T, blank=0):
    """枚举所有长度=T的路径，统计有多少条压缩后等于 target_labels。
    
    Args:
        target_labels: list[int]，目标标签序列（不含 blank）
        T: 路径长度（帧数）
        blank: blank 符号的 token id
    
    Returns:
        count: 合法路径数
        example_paths: 前10条合法路径的例子
    """
    V = len(set(target_labels) | {blank}) + 1  # 词表大小猜测（会扩展）
    valid_paths = []
    
    # 枚举所有可能的路径
    for path in product(range(V + 5), repeat=T):  # 粗暴但有效
        if collapse(list(path), blank=blank) == target_labels:
            valid_paths.append(path)
    
    return len(valid_paths), valid_paths[:10]

# 示例
target = [1, 2]  # a, b
T_test = 4
count, examples = count_ctc_paths(target, T_test, blank=0)
print(f"目标 = {target}，帧数 T = {T_test}")
print(f"合法路径总数 = {count}")
print(f"前5条路径的例子：")
for ex in examples[:5]:
    print(f"  {list(ex)} → {collapse(list(ex))}")

print(f"\n比较：")
print(f"总路径数上界 (V^T，词表V=3) = 3^{T_test} = {3**T_test}")
print(f"实际合法路径 = {count}")
print(f"比例 = {count / (3**T_test):.1%}")


In [ ]:
def collapse(path, blank=0):
    """CTC 路径压缩：先去相邻重复，再去 blank。
    
    【简洁版本】用列表推导式
    - 如果看不懂列表推导式，可以看上一个 cell 的 collapse_with_loop 版本，逻辑完全一样
    - 两个版本等价，这里只是更紧凑的写法
    """
    # 步骤1：去相邻重复
    # 含义：[p for i, p in enumerate(path) if i == 0 or p != path[i-1]]
    #      = 保留第0个元素，或保留与前一个不同的元素
    deduped = [p for i, p in enumerate(path) if i == 0 or p != path[i-1]]
    
    # 步骤2：去 blank
    # 含义：[p for p in deduped if p != blank]
    #      = 保留所有不是 blank 的元素
    return [p for p in deduped if p != blank]

# 示例验证
examples = [
    ([1, 1, 0, 2, 2, 0, 3], 'a a ∅ b b ∅ c'),
    ([1, 0, 1, 2],           'a ∅ a b'),
    ([1, 1, 2],              'a a b'),
    ([0, 0, 1, 0],           '∅ ∅ a ∅'),
]
for path, desc in examples:
    result = collapse(path)
    print(f'  [{desc}]  →  {result}')

## 3. 贪婪解码（Greedy Decode）

精确 CTC 解码需要 beam search（枚举所有路径太慢）。
**贪婪解码**是最快的近似：每帧直接取 argmax，得到一条路径，再执行 collapse。

```
π_greedy = [argmax p(token | t)  for t in 1..T]
y_greedy = collapse(π_greedy)
```

贪婪解码不是最优的——它只考虑了最高概率的单条路径，而正确答案可能由多条次优路径联合撑起来。
但在 Whisper、DeepSpeech 等实践中，它作为快速 baseline 效果已经不错。

In [ ]:
# 演示贪婪解码
np.random.seed(42)
T_demo, V_demo = 7, 5  # 7帧，词表大小5（0=blank, 1=a, 2=b, 3=c, 4=d）
logits_demo = np.random.randn(T_demo, V_demo)
# 手动拉高某些帧让结果更直观
logits_demo[0, 1] = 5   # 帧0: a
logits_demo[1, 0] = 5   # 帧1: blank
logits_demo[2, 0] = 5   # 帧2: blank
logits_demo[3, 2] = 5   # 帧3: b
logits_demo[4, 2] = 5   # 帧4: b
logits_demo[5, 0] = 5   # 帧5: blank
logits_demo[6, 3] = 5   # 帧6: c

preds = np.argmax(logits_demo, axis=1)
vocab = {0:'∅', 1:'a', 2:'b', 3:'c', 4:'d'}
print('帧级 argmax  :', [vocab[p] for p in preds])
print('去相邻重复  :', [vocab[p] for i,p in enumerate(preds) if i==0 or p!=preds[i-1]])
decoded = collapse(preds.tolist())
print('最终解码结果:', [vocab[p] for p in decoded], '→ token ids:', decoded)


## 4. ✏️ 实现 `ctc_greedy_decode(logits, blank=0)`

**签名**：
```python
def ctc_greedy_decode(logits: np.ndarray, blank: int = 0) -> list:
    # logits: (T, V) numpy 数组（未经 softmax 的原始得分）
    # 返回: list[int]，压缩后的 token id 序列（不含 blank）
```

**3步实现**：

| 步骤 | 操作 | 工具 |
|---|---|---|
| 1 | `path = logits.argmax(axis=1)` | 每帧取最大 token |
| 2 | 去相邻重复：`[a,a,b,b] → [a,b]` | 循环或 `itertools.groupby` |
| 3 | 去 blank：过滤掉 blank token | list comprehension |

**验收标准**：
- 全 blank logits → `[]`（空列表）
- `[a, a, ∅, b, ∅]` → `[a, b]`
- `[a, ∅, a]` → `[a, a]`（两个 a 中间有 blank，不合并）

### 注意：logits vs log_probs

在之前的示例中，我们手动构造了 `log_probs = np.log(概率)`（概率的 log）。
但真实的神经网络模型输出的是 **logits**（未经 softmax 的原始得分），不是概率。

```
logits（神经网络原始输出）: [5.2, -1.3, 0.8]   （任意实数，可正可负）
                            ↓ softmax
概率（概率分布）:           [0.98, 0.01, 0.01]  （和=1，都在0-1）
                            ↓ log
log 概率（对数概率）:       [-0.02, -4.6, -4.8] （负数或0）
```

**ctc_greedy_decode 接收的是 logits（原始得分），不是 log 概率**。
为什么 argmax 对两者都成立？因为 `softmax` 和 `log` 都是**单调递增**的：

```
argmax(logits) = argmax(softmax(logits)) = argmax(log(softmax(logits)))

所以 logits 的最大位置 = 概率的最大位置 = log概率的最大位置
```

你不需要显式转换，直接对 logits 取 argmax 就行。

### 预告：L69 会用到 log 域和 logaddexp

当计算**多路径的概率之和**时，会遇到数值问题：

```python
# 直接相乘会发生数值下溢（underflow）
p1, p2 = 1e-100, 1e-100
total = p1 + p2  # 在 log 空间里需要 log 加法

# L69 的解法：用 log 空间
log_p1, log_p2 = np.log(p1), np.log(p2)
log_total = np.logaddexp(log_p1, log_p2)  # log 下的加法公式
# np.logaddexp(a, b) = log(exp(a) + exp(b))，避免溢出
```

现在你只需要知道贪婪解码不涉及这个复杂性（不用求和），L69 会详细讲。

### 贪婪解码什么时候会出错？

**贪婪的定义**：每一帧都独立取概率最高的 token，然后 collapse。

**何时失效**：当最优的目标序列由多条次优路径联合撑起来，而贪婪只看一条路径时，贪婪的答案概率可能远低于最优值。

**具体数字例子**：

```
target = [a, b]，3帧
词表 = {0: ∅, 1: a, 2: b}

logits 矩阵：           概率（softmax后，近似）
帧0：[5.0, 0.1, 0.1]  → token-0(∅) = 99%   token-1(a) = 0.5%
帧1：[5.0, 0.1, 0.1]  → token-0(∅) = 99%   token-1(a) = 0.5%  token-2(b) = 0.5%
帧2：[0.1, 0.1, 5.0]  → token-2(b) = 99%

【贪婪解码】：
帧0: argmax = 0(∅)
帧1: argmax = 0(∅)
帧2: argmax = 2(b)
路径 = [∅, ∅, b] → collapse → [b]   ✗ 错错错！期望[a,b]
贪婪路径概率 = 0.99 × 0.99 × 0.99 ≈ 0.97

【更好的路径】：
路径1 = [a, ∅, b] → collapse → [a, b]  ✓ 正确！
        P = 0.005 × 0.99 × 0.99 ≈ 0.0049

路径2 = [a, a, b] → collapse → [a, b]  ✓ 正确！
        P = 0.005 × 0.005 × 0.99 ≈ 0.0000247

两条正确路径概率之和 ≈ 0.0049 + 0.000025 ≈ 0.005

所以虽然贪婪路径（[∅,∅,b]）个体概率最高（0.97），
但两条正确路径合起来才是真正的答案（[a,b]的总概率 ≈ 0.005）。
贪婪会错误地输出[b]而不是[a,b]。
```

**结论**：贪婪解码快但不保证最优。Beam search（探索多条高概率路径并合并）或完整前向算法（L69）才能给出精确答案。
在实践中，Whisper/DeepSpeech 对训练好的模型用贪婪也效果不错（因为模型学习后，正确答案的路径概率会被拉高）。

In [ ]:
def ctc_greedy_decode(logits: np.ndarray, blank: int = 0) -> list:
    """CTC 贪婪解码：argmax 每帧，去相邻重复，去 blank。

    Args:
        logits: shape (T, V) numpy 数组（未经 softmax 的原始得分）
        blank:  blank 符号的 token id，默认 0
    Returns:
        解码后的 token id 列表
    """
    # ✏️ TODO: 第1步——每帧取 argmax
    preds = None

    # ✏️ TODO: 第2步——去相邻重复（保留第一个，后续只保留与前一个不同的）
    deduped = None

    # ✏️ TODO: 第3步——去掉 blank，返回 list[int]
    raise NotImplementedError("请先完成上方三个 TODO 步骤，再运行检验 cell；卡住可看 solutions/L68_ctc_alignment_solutions.md")

In [ ]:
# --- 检查 ctc_greedy_decode ---

V_check = 5
logits_check = np.full((7, V_check), -10.0)
forced_ids = [1, 0, 0, 2, 2, 0, 3]
for t, tok in enumerate(forced_ids):
    logits_check[t, tok] = 10.0

try:
    result = ctc_greedy_decode(logits_check, blank=0)
    assert result == [1, 2, 3], f'期望 [1,2,3]，实际得到 {result}'
    print('✅ ctc_greedy_decode([a ∅ ∅ b b ∅ c]) =', result, '= [a, b, c]')

    # 边界：全 blank 输入 → 空列表
    logits_blank = np.zeros((5, V_check))
    logits_blank[:, 0] = 10.0
    assert ctc_greedy_decode(logits_blank) == [], '全 blank 应返回空列表'
    print('✅ 全 blank 输入 → []')

    # 边界：单帧单字符
    logits_single = np.array([[-1.0, 5.0, -1.0]])
    assert ctc_greedy_decode(logits_single) == [1]
    print('✅ 单帧 argmax=1 → [1]')

except (NotImplementedError, TypeError):
    print('⬜ 请先实现 ctc_greedy_decode 的三个 TODO 步骤，再运行此检验 cell')


## 5. 参数实验：随机 logits 下的序列压缩率

**实验设置**：
- `T = 50`（输入帧数），`vocab_size = 30`（含 blank=0）
- 跑 1000 次，每次随机采样 `logits ~ N(0,1)`
- 统计 `len(decoded) / T` 的均值和分布

**预期现象**：
- 每帧随机时，blank 被选中概率约 `1/30 ≈ 3.3%`，相邻重复概率约 `1/30`
- 去重+去blank 后，平均输出长度约为输入的 `~93-94%`
  - 推导：每帧选到与前一帧相同 token 的概率约 1/V，去重后存活率约 (V-1)/V；
    再去 blank（选中概率 1/V），两步独立近似后，
    期望压缩率 ≈ ((V-1)/V)² = (29/30)² ≈ 0.934
- 实际语音模型训练后 blank 被大量预测，压缩率会**远低于**此随机基线（blank 多则输出更短）

## 压缩率公式推导：((V-1)/V)² 是怎么来的？

**实验问题**：随机 logits（每帧随机选 token）的情况下，经过"去重复+去blank"后，输出序列有多短？

**假设前提**：
- 词表大小 V（含 blank），随机时每个 token 被选中的概率都是 1/V
- 去重复和去 blank 两个操作**独立**（关键假设，后面会讨论）

**推导**：

### 第一步：去相邻重复的存活率

当前帧 token 与前一帧不同的概率？

```
前一帧选了某个 token A（概率 1/V）
当前帧选了不同 token 的概率 = 1 - P(当前帧=A)
                    = 1 - 1/V
                    = (V-1)/V
```

直觉：V 个选择里，当前帧选到"与前一帧不同"的有 V-1 个。

**例**：V=30（blank+29个字符），P(与前一帧不同) = 29/30 ≈ 0.967

### 第二步：去 blank 的存活率

当前帧**不是 blank** 的概率？

```
P(token ≠ blank) = 1 - P(token = blank)
                 = 1 - 1/V
                 = (V-1)/V
```

同样的逻辑：V 个选择里，非 blank 的有 V-1 个。

### 第三步：两步**独立近似**后的总存活率

假设一帧活过去（去重复）的事件 A 和这帧本身不是 blank 的事件 B 是**独立**的，那么：

```
P(一帧活过两步) = P(A) × P(B)
               = (V-1)/V × (V-1)/V
               = ((V-1)/V)²
```

期望输出长度 = 输入长度 T × ((V-1)/V)²

**验证数字**：V=30
```
期望压缩率 = (29/30)² = 0.9667² ≈ 0.9344 ≈ 93.4%

意思是：50 帧输入，期望输出约 50 × 0.934 ≈ 46.7 帧
```

### ⚠️ 独立性假设的讨论

"两步独立"严格来说不完全对。在随机模型中：
- **第一步**（去重复）："当前帧≠前一帧"的概率是 (V-1)/V
- **第二步**（去 blank）："当前帧≠blank"的概率是 (V-1)/V

严格来说，这两个事件有隐含关联——比如，"当前帧是 blank"意味着"当前帧≠(非 blank 的前一帧)"。
但当 V 很大（如 V=30）时，这种关联很弱，独立近似就很准（相对误差 < 2%）。

对于严格证明，需要用联合概率：
```
P(活过两步) = P(当前≠前一帧 AND 当前≠blank)

随机情况下，当前帧有 V 个等可能选择。前一帧是某个 token，blank 是另一个 token（或同一个）：
- 如果前一帧≠blank：有 V-2 个选择（排除前一帧和blank）→ P = (V-2)/V
- 如果前一帧=blank：有 V-1 个选择（排除blank/前一帧）→ P = (V-1)/V

前一帧是 blank 的概率 = 1/V；不是 blank 的概率 = (V-1)/V
联合：P = (V-1)/V × (V-2)/V + 1/V × (V-1)/V = (V-1)/V × ((V-2)/V + 1/V) = (V-1)/V × (V-1)/V = ((V-1)/V)²

咦，算出来一样！所以独立近似对随机情况其实是**精确的**。
```

**结论**：公式 ((V-1)/V)² 在随机 logits 下是精确的；在实际训练后，blank 会大量被预测，压缩率会更低。

In [ ]:
try:
    np.random.seed(0)
    T_exp, V_exp, N = 50, 30, 1000

    ratios = []
    for _ in range(N):
        logits_exp = np.random.randn(T_exp, V_exp)
        decoded_exp = ctc_greedy_decode(logits_exp, blank=0)
        ratios.append(len(decoded_exp) / T_exp)

    ratios = np.array(ratios)
    print(f'输入长度 T = {T_exp}，词表大小 V = {V_exp}（含blank）')
    print(f'1000次随机实验：')
    print(f'  平均输出长度     : {ratios.mean()*T_exp:.1f} 帧')
    print(f'  平均压缩率 len/T : {ratios.mean():.3f}')
    print(f'  最小/最大压缩率  : {ratios.min():.3f} / {ratios.max():.3f}')
    print()
    expected = ((V_exp - 1) / V_exp) ** 2
    print(f'期望压缩率公式 ((V-1)/V)² = ({V_exp-1}/{V_exp})² ≈ {expected:.3f}')
    print('（双步独立近似：先去相邻重复，再去 blank；随机 logits 下两步同量级）')

except (NotImplementedError, TypeError):
    print('⬜ 请先实现 ctc_greedy_decode，再运行参数实验')


## 6. 补充：CTC 的单调路径约束（Monotonic Paths）

**约束定义**：CTC 路径 π 必须满足：处理帧的顺序从左到右单调，标签（非 blank 符号）的出现顺序也严格单调向前——不能倒退。

```
正确的单调路径：
  帧0 → a（第1个标签）
  帧1 → a（还是第1个标签，blank 跳过不算）
  帧2 → blank
  帧3 → b（第2个标签，向前推进）
  帧4 → b（还是第2个标签）
  ✓ 标签序列是 a → a → b，严格非递减

错误的非单调路径（CTC 算法不允许）：
  帧0 → a
  帧1 → b
  帧2 → a  ✗ 倒退到之前的标签了！CTC 禁止
```

**为什么需要这个约束？**

从物理上讲，语音是时间序列，字符的发音按字典顺序从左到右进行。
你不可能先说 "o" 再回头说 "h"（hello 的 h 必须最先出现）。
CTC 把这个事实写成算法约束：标签轴不倒退，确保对齐与语音时序一致。

**在贪婪解码中，单调性是自动满足的吗？**

是的！原因很简单：
- 贪婪解码从帧 0 逐个处理到帧 T-1
- collapse 只做"去重复"和"去 blank"，**不会改变顺序**
- 所以路径中的标签顺序必然保持从左到右

```
例如：
帧序列  [a, blank, b, b, blank, c]
         ↓ collapse（不改变顺序）
输出    [a, b, c]   ✓ 单调

不可能出现 [c, a, b] 这样的倒序，因为 collapse 是有序的过滤操作
```

**在 L69 前向算法中，单调性为什么需要显式建模？**

精确 CTC 算法要**枚举并求和所有合法路径**。不是所有 V^T 条路径都是合法的——必须挑出满足单调约束的那些。
前向算法用动态规划，显式追踪"当前处理到帧 t、已展开到标签序列的第 s 个位置"，自动确保单调性。
如果不加这个约束，路径空间会大得多，计算量爆炸。

**总结**：单调路径约束是 CTC 的**基本假设**，不是可选的。贪婪解码自动满足它；精确算法显式利用它来压缩搜索空间。

## 附录 C · 从贪婪解码到前向算法：L69 预习（8 分钟）

1. **L68 做了什么**：找**一条**最优路径（贪婪近似）。
2. **L69 要做什么**：对**所有**合法路径的概率**求和**（精确训练损失）。
3. **为何需要 DP**：路径数 O(V^T)；DP 降到 O(T·S)，其中 `S = 2L + 1`。
4. **与 L67 的关系**：编辑距离 DP 也是“填表递推”；CTC 只是把表里的值换成 log 概率。
5. **log 域一句**：概率相乘 → log 域用 `logaddexp` 做“log 下的加法”。

```text
        s=0   s=1   s=2   …   (扩展标签轴)
t=0     α     ·     ·
t=1     ·     ·     ·
…
```

In [ ]:
from itertools import product

toy_labels = [1, 2]  # a b
toy_paths = [
    path
    for path in product(range(3), repeat=3)
    if collapse(path, blank=0) == toy_labels
]
print("T=3 时的合法路径数：", len(toy_paths))
print("前 5 条路径：", toy_paths[:5])
print("精确求和要等 L69 的 ctc_forward；L68 这里只先看路径空间怎么长出来。")

## 本课收束

本节实现了 `ctc_greedy_decode`，它输出一个 `list[int]`（压缩后的 token id 序列），对应 CTC 路径的近似最优解。
该函数直接服务于 `aurora.speech` 推理管线——在 `torch.nn.CTCLoss` 训练完成后，贪婪解码是最快的线上解码方式。
本课附录 C 已把 L69 的前向算法预热：先看路径空间，再看 log 域递推，就不会在下一节突然掉进 cliff。
下一节（L69）将实现 CTC 前向算法的完整动态规划，追踪 blank-skip 和标签折叠两条合法路径，计算序列概率。
L69 之后还将讨论 Whisper 选择交叉熵而非 CTC 的原因。

## ✏️ 白板挑战：CTC 解码手推（目标 10 分钟）

盖上屏幕，纸上推导：

**问 1**：路径 `[a, ∅, a, a, ∅, b]` 压缩成什么目标序列？  
（去相邻重复: [a,∅,a,∅,b]，去blank: [a,a,b]）

**问 2**：目标序列 `"ab"` 对应哪些长度=4的 CTC 路径？（blank=0, a=1, b=2）  
（需要先有a序列再b序列，中间可插blank；如[a,a,∅,b],[a,∅,b,b],[∅,a,b,b],[a,∅,∅,b]等）

**问 3**：贪婪解码与最优解码的区别是什么？什么情况下贪婪解码会出错？  
（贪婪=每帧argmax；当最优路径需要"当前帧选次优token，下帧选最优"时贪婪失效，beam search更好）

**问 4**：CTC 为什么不需要帧级标注（forced alignment）？  
（总概率=所有合法路径的概率和；只需序列级标注"这段音频对应这句话"即可训练）

**问 5**：若 logits shape=(50, 30)，贪婪解码后最长可能输出多少个 token？  
（最多50个，若每帧都是不同非blank token；最少0个，若全是blank或全是同一token）

推导完成后运行下方格验证。

In [ ]:
# ✏️ 对答案格
import numpy as np

# 问1：路径压缩
path_q1 = [1, 0, 1, 1, 0, 2]  # a=1,blank=0,b=2
result_q1 = collapse(path_q1, blank=0)
assert result_q1 == [1, 1, 2], f"得到{result_q1}，期望[1,1,2]"
print(f"Q1 ✅  [a,∅,a,a,∅,b]压缩后={result_q1} = [a,a,b]（两个a中间有blank，不合并）")

# 问2：长度=4路径枚举（验证collapse输出正确）
target_q2 = [1, 2]  # ab
valid_paths = []
for p0 in range(3):
    for p1 in range(3):
        for p2 in range(3):
            for p3 in range(3):
                path = [p0,p1,p2,p3]
                if collapse(path, blank=0) == target_q2:
                    valid_paths.append(path)
assert len(valid_paths) >= 4, f"找到{len(valid_paths)}条路径，应≥4"
print(f"Q2 ✅  目标'ab'对应{len(valid_paths)}条长度=4的CTC路径（如{valid_paths[:3]}...）")

# 问3：贪婪 vs 最优（演示场景）
try:
    # 构造一个贪婪次优的logits
    logits_q3 = np.zeros((3, 3))
    logits_q3[0, 0] = 1.0   # 第0帧：blank最优
    logits_q3[1, 1] = 0.9   # 第1帧：token1次优
    logits_q3[1, 0] = 1.0   # 第1帧：blank最优
    logits_q3[2, 2] = 1.0   # 第2帧：token2最优
    greedy_path = logits_q3.argmax(axis=1)
    greedy_result = collapse(greedy_path.tolist(), blank=0)
    print(f"Q3 ✅  贪婪解码路径={greedy_path.tolist()} → {greedy_result}")
    print(f"       beam search可能找到更好路径（此例中贪婪=最优，复杂情况下不同）")
except Exception as e:
    print(f"Q3 ✅  贪婪=每帧argmax；beam search探索多路径，更准确但更慢")

# 问4：序列级训练（概念验证 — 使用ctc_greedy_decode）
try:
    logits_q4 = np.array([[0.1, 5.0, 0.1],  # 帧0: token1最大
                           [0.1, 0.1, 5.0],  # 帧1: token2最大
                           [5.0, 0.1, 0.1]]) # 帧2: blank最大
    decoded = ctc_greedy_decode(logits_q4, blank=0)
    assert decoded == [1, 2], f"解码结果={decoded}"
    print(f"Q4 ✅  ctc_greedy_decode: logits→{decoded}（只需序列标注[1,2]，无需帧对齐）")
except (NotImplementedError, TypeError):
    print(f"Q4 ✅  CTC不需帧级标注：优化所有合法路径的概率总和（待实现后验证）")

# 问5：最长/最短输出
try:
    T_q5, V_q5 = 10, 5
    # 最长：每帧交替不同非blank token
    logits_max = np.zeros((T_q5, V_q5))
    for t in range(T_q5): logits_max[t, (t % (V_q5-1)) + 1] = 1.0  # 非blank交替
    decoded_max = ctc_greedy_decode(logits_max, blank=0)
    # 最短：全blank
    logits_min = np.zeros((T_q5, V_q5)); logits_min[:, 0] = 1.0
    decoded_min = ctc_greedy_decode(logits_min, blank=0)
    print(f"Q5 ✅  T=10时: 最多输出{len(decoded_max)}个token，最少{len(decoded_min)}个（全blank）")
except (NotImplementedError, TypeError):
    print(f"Q5 ✅  T帧最多T个token（无相邻重复），最少0个（全blank）（待实现后验证）")
print("\n🎉 CTC解码白板挑战通过！")

In [ ]:
# ✏️ 本课自评
l68_review = {
    "ctc_blank_understood":      None,  # 理解blank符号的作用：允许帧数>>字符数，支持字符重复？True/False
    "collapse_rules":            None,  # 记住压缩2步：去相邻重复→去blank？True/False
    "greedy_decode_impl":        None,  # ctc_greedy_decode()实现正确，3个测试用例通过？True/False
    "no_alignment_needed":       None,  # 理解CTC为什么不需要帧级对齐标注？True/False
    "whiteboard_passed":         None,  # 白板挑战5道手推全部完成？True/False
}

unfilled = [k for k, v in l68_review.items() if v is None]
assert not unfilled, f'还未填写：{unfilled}'
weak = [k for k, v in l68_review.items() if v is False]
if weak:
    print(f'⚠️  需要加强：{weak}')
else:
    print('✅ L68 全部通关！进入 L69：CTC 前向算法')

---

→ **下一课**　[L69 · CTC 前向算法](L69_ctc_forward.ipynb)

> 下节课将学习 **CTC 前向算法**：所有路径概率求和，O(T·S) 纯 NumPy 实现。先读本课附录 C，再回到 L67 的 DP 填表逻辑。